# 04 — Figures

Generates all figure panels for the Mouse vs AI benchmark paper.

**Main panels** (`figures/main/`):
- `fig3b_mouse_vs_ai_heldout_hist` — AI held-out performance distribution vs mice
- `fig3c_architecture_heldout` — Per-backbone held-out performance
- `fig3d_track1_track2_scatter_winners_heldout_thresh` — T1 × T2 scatter (held-out ≥ 0.5 filter)
- `fig3d_track1_track2_scatter_winners_normal_thresh` — T1 × T2 scatter (Normal ≥ 0.5 filter)

**Supplementary panels** (`figures/supplement/`):
- `supp_track1_track2_all_scatters` — All 5 perturbations vs Track 2
- `supp_recurrence_effect` — Recurrent vs non-recurrent models
- `supp_track1_track2_scatter_winners_allrounders_heldout_thresh` — T1 × T2 with all-rounders (held-out ≥ 0.5)
- `supp_track1_track2_scatter_winners_allrounders_normal_thresh` — T1 × T2 with all-rounders (Normal ≥ 0.5)
- `supp_track1_track2_scatter_arch` — T1 × T2 colored by architecture (held-out ≥ 0.5)
- `supp_track1_track2_scatter_arch_normal_thr` — T1 × T2 colored by architecture (Normal ≥ 0.5)
- `supp_model_property_trends` — 2 × 3 composite: model properties (parameters, depth) vs T1 and T2

**Before running:** place the four data CSVs in `data/` (see required-files cell below).  
**Runtime:** < 30 s. No GPU required.  
Run *Kernel → Restart & Run All*.

## 1. Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.colors import LogNorm
from scipy import stats


## 2. Global figure style and helpers

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
ROOT     = Path().resolve().parent   # src/ -> repo root
DATA_DIR = ROOT / 'data'
FIG_DIR  = ROOT / 'figures'
(FIG_DIR / 'main').mkdir(parents=True, exist_ok=True)
(FIG_DIR / 'supplement').mkdir(parents=True, exist_ok=True)

# ── Sizes ─────────────────────────────────────────────────────────────────
CM = 1 / 2.54   # cm → inches

# ── Figure style ──────────────────────────────────────────────────────────
# font.size=7 per NeurIPS guidelines; pdf.fonttype=42 keeps text editable.
PAPER_RC = {
    'font.size':        7,
    'axes.titlesize':   8,
    'axes.labelsize':   7,
    'xtick.labelsize':  6,
    'ytick.labelsize':  6,
    'legend.fontsize':  6,
    'axes.linewidth':   0.8,
    'lines.linewidth':  1.2,
    'patch.linewidth':  0.5,
    'figure.dpi':       110,
    'savefig.dpi':      300,
    'pdf.fonttype':     42,
    'ps.fonttype':      42,
}

# ── Colours ───────────────────────────────────────────────────────────────
COLOR_AI          = '#4C72B0'
COLOR_MOUSE       = '#DD8452'
COLOR_NEUTRAL     = '#2E2E2E'
COLOR_GRAY        = '#7F7F7F'
COLOR_RECURRENCE  = '#7E57C2'

# ── Track saved paths for summary cell ────────────────────────────────────
saved_files = []


# ── Helpers ───────────────────────────────────────────────────────────────

def require_file(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Required file not found: {p.resolve()}')
    return p


def save_fig(fig, name, subdir='main'):
    """Save PDF + PNG at exact figsize (artboard preserved for Illustrator)."""
    dest = FIG_DIR / subdir
    dest.mkdir(parents=True, exist_ok=True)
    for ext in ('pdf', 'png'):
        path = dest / f'{name}.{ext}'
        dpi  = 200 if ext == 'png' else None
        fig.savefig(path, facecolor='white', **(dict(dpi=dpi) if dpi else {}))
        saved_files.append(str(path))
    print(f'  saved: {subdir}/{name}.pdf + .png')


def style_ax(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=3, pad=2)


def max4_ticks(ax):
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))


def corr_annotation(ax, r, p, n, loc='upper left'):
    txt = f'r = {r:.3f}\np = {p:.3g}\nn = {n}'
    coords = {
        'upper left':  (0.04, 0.96, 'left',  'top'),
        'upper right': (0.96, 0.96, 'right', 'top'),
        'lower left':  (0.04, 0.04, 'left',  'bottom'),
        'lower right': (0.96, 0.04, 'right', 'bottom'),
    }
    x, y, ha, va = coords[loc]
    ax.text(x, y, txt, transform=ax.transAxes, ha=ha, va=va, fontsize=6,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      alpha=0.85, edgecolor='lightgray', linewidth=0.5))


def lighten_color(hex_col, factor):
    r = int(hex_col[1:3], 16)
    g = int(hex_col[3:5], 16)
    b = int(hex_col[5:7], 16)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'#{r:02X}{g:02X}{b:02X}'


def top_owners(df, sort_col, max_n=3):
    return (df.sort_values(sort_col, ascending=False)
              .drop_duplicates('owner')
              .head(max_n)
              .reset_index(drop=True))


## 3. Data loading

**Required files** — place in `data/` before running:

| File | Contents |
|------|----------|
| `leaderboard_track1.csv` | Per-condition success rates for all submissions (Normal, Clutter, Contrast, Fog, RDK) plus computed robustness metrics (`heldout_performance`, `performance_drop`, `robustness_ratio`) |
| `leaderboard_track2.csv` | Neural alignment scores for evaluated submissions. `track2_score` is the primary metric: mean Pearson r over visually-tuned neurons (C0.005 filter: firing rate ≥ 100 spikes and visual responsiveness R_visual ≥ 0.005) |
| `model_metadata.csv` | Architecture metadata: backbone family, number of layers, number of parameters, recurrence, RL algorithm |
| `mouse_performance.csv` | Behavioural performance of 5 mouse recording sessions across the same 5 conditions — used as the biological reference |

In [ ]:
PERTURBATIONS = ['Normal', 'Clutter', 'Contrast', 'Fog', 'RDK']
HELDOUT_COLS  = ['Clutter', 'Contrast', 'RDK']   # held-out mean

# Primary Track-2 metric (column name in leaderboard_track2.csv)
T2_PRIMARY = 'track2_score'

# ── Load CSVs ──────────────────────────────────────────────────────────────────────────────────────
track1_df = pd.read_csv(require_file(DATA_DIR / 'leaderboard_track1.csv'))
track2_df = pd.read_csv(require_file(DATA_DIR / 'leaderboard_track2.csv'))
arch_df   = pd.read_csv(require_file(DATA_DIR / 'model_metadata.csv'))
mouse_raw = pd.read_csv(require_file(DATA_DIR / 'mouse_performance.csv'))

# Schema normalisation: new-format T2 file uses model_name instead of id
# (baselines have no integer prefix and receive id = NaN)
if 'model_name' in track2_df.columns and 'id' not in track2_df.columns:
    track2_df = track2_df.rename(columns={'model_name': 'model'})
    track2_df['id'] = pd.to_numeric(
        track2_df['model'].str.extract(r'^(\d+)_')[0], errors='coerce'
    )

# Consistent id type
for _df in (track1_df, track2_df, arch_df):
    _df['id'] = _df['id'].astype(str)

# ── Validate required columns ────────────────────────────────────────────────────────────────────────────────────
_req = {
    'leaderboard_track1.csv': {'id', 'owner'} | set(PERTURBATIONS),
    'leaderboard_track2.csv': {'track2_score'},    # id added by normalisation above
    'model_metadata.csv':     {'id', 'backbone', 'num_layers',
                                'num_params', 'has_recurrence'},
    'mouse_performance.csv':  {'subject_id'} | set(PERTURBATIONS),
}
for _fname, _req_cols in _req.items():
    _actual = set(pd.read_csv(DATA_DIR / _fname, nrows=0).columns)
    _miss   = _req_cols - _actual
    if _miss:
        raise ValueError(f'{_fname} is missing required columns: {_miss}')

print(f'  leaderboard_track1.csv : {len(track1_df):4d} models')
print(f'  leaderboard_track2.csv : {len(track2_df):4d} models')
print(f'  model_metadata.csv     : {len(arch_df):4d} models')
print(f'  mouse_performance.csv  : {len(mouse_raw):4d} sessions')
print(f'  Track1 ∩ Track2        : '
      f'{len(set(track1_df["id"]) & set(track2_df["id"]))} models')

## 4. Compute metrics

In [ ]:
# Mouse session data — loaded from mouse_performance.csv
_mouse_raw = (mouse_raw[['subject_id'] + PERTURBATIONS]
              .rename(columns={'subject_id': 'id'}))


def compute_metrics(df, heldout_cols=None):
    """Append five robustness metric columns (returns a copy).

    Columns added:
      normal_performance  = Normal score
      heldout_performance = mean(Clutter, Contrast, RDK)
      robustness_ratio    = heldout / normal
      performance_drop    = normal - heldout
      normalized_drop     = (normal - heldout) / normal
    """
    if heldout_cols is None:
        heldout_cols = HELDOUT_COLS
    out = df.copy()
    out['normal_performance']  = out['Normal']
    out['heldout_performance'] = out[heldout_cols].mean(axis=1)
    safe = out['normal_performance'].where(out['normal_performance'] != 0)
    out['robustness_ratio'] = out['heldout_performance'] / safe
    out['performance_drop'] = out['normal_performance'] - out['heldout_performance']
    out['normalized_drop']  = (out['normal_performance'] - out['heldout_performance']) / safe
    return out


ai_df    = compute_metrics(track1_df)
mouse_df = compute_metrics(_mouse_raw)

METRIC_COLS = ['normal_performance', 'heldout_performance',
               'robustness_ratio', 'performance_drop', 'normalized_drop']

for label, df_ in [(f'AI (n={len(ai_df)})', ai_df),
                   (f'Mouse (n={len(mouse_df)})', mouse_df)]:
    print(f'\n{label}:')
    rows = []
    for m in METRIC_COLS:
        s = df_[m]
        rows.append([m, f'{s.mean():.4f}', f'{s.std():.4f}',
                     f'{s.min():.4f}', f'{s.max():.4f}', int(s.isna().sum())])
    print(pd.DataFrame(rows,
          columns=['metric', 'mean', 'std', 'min', 'max', 'n_nan']).to_string(index=False))

## 5. Architecture and owner setup

In [ ]:
# ── Architecture merge ────────────────────────────────────────────────────
_arch_cols = ['id', 'backbone', 'num_layers', 'num_params', 'has_recurrence', 'policy_algo']
_arch_cols = [c for c in _arch_cols if c in arch_df.columns]
ai_arch = ai_df.merge(arch_df[_arch_cols].copy(), on='id', how='inner')

print(f'Models with architecture info: {len(ai_arch)}')
print('\nBackbone counts:')
bb_counts = ai_arch['backbone'].value_counts()
print(bb_counts.to_string())

# ── Recurrence as boolean ─────────────────────────────────────────────────
arch_rec = arch_df.copy()
arch_rec['has_recurrence_bool'] = (
    arch_rec['has_recurrence'].astype(str).str.strip().str.lower()
    .map({'true': True, '1': True, 'yes': True,
          'false': False, '0': False, 'no': False}))
ai_rec = ai_df.merge(
    arch_rec[['id', 'has_recurrence_bool']], on='id', how='inner'
).dropna(subset=['has_recurrence_bool'])
ai_rec['has_recurrence_bool'] = ai_rec['has_recurrence_bool'].astype(bool)
print(f'\nRecurrence: no={int((~ai_rec["has_recurrence_bool"]).sum())}  '
      f'yes={int(ai_rec["has_recurrence_bool"].sum())}')

# ── Owner map — 'owner' column is already in leaderboard_track1.csv ───────
lb_owner_map = (track1_df.dropna(subset=['owner'])
                          .drop_duplicates('id')
                          .set_index('id')['owner'])

# ── Top-3 AI by held-out performance (reference) ──────────────────────────
top3_heldout = (
    ai_df.dropna(subset=['heldout_performance'])
         .sort_values('heldout_performance', ascending=False)
         .head(3).copy())
top3_heldout['owner'] = top3_heldout['id'].map(lb_owner_map).fillna('?')
print('\nTop 3 AI by held-out performance:')
print(top3_heldout[['id', 'owner', 'Normal', 'heldout_performance']].to_string(index=False))

# ── Combined T1+T2+arch frames ────────────────────────────────────────────
# Two threshold variants:
#   full_df        — heldout_performance > T1_THRESHOLD   (held-out filter)
#   full_df_normal — Normal              > T1_NORMAL_THRESHOLD  (normal filter)
T1_THRESHOLD        = 0.5   # applied to heldout_performance
T1_NORMAL_THRESHOLD = 0.5   # applied to Normal (unperturbed) performance

ai_df_thr    = ai_df[ai_df['heldout_performance'] > T1_THRESHOLD].copy()
ai_df_normal = ai_df[ai_df['Normal']              > T1_NORMAL_THRESHOLD].copy()
print(f'\nT1 filter (heldout > {T1_THRESHOLD}):  {len(ai_df_thr):3d} / {len(ai_df)} models')
print(f'T1 filter (Normal  > {T1_NORMAL_THRESHOLD}):  {len(ai_df_normal):3d} / {len(ai_df)} models')


def _build_full_frame(ai_subset, t2_col=T2_PRIMARY, t2_df=None):
    """Inner-merge T1 + T2 + arch; add owner + min-max normalised scores."""
    if t2_df is None:
        t2_df = track2_df
    t2_keep   = [c for c in ['id', t2_col] if c in t2_df.columns]
    arch_keep = [c for c in ['id', 'backbone', 'num_params', 'num_layers',
                              'has_recurrence', 'policy_algo']
                 if c in arch_df.columns]
    merged = (ai_subset
              .merge(t2_df[t2_keep],    on='id', how='inner')
              .merge(arch_df[arch_keep], on='id', how='inner')
              .dropna(subset=['heldout_performance', t2_col])
              .copy())
    merged['owner'] = merged['id'].map(lb_owner_map).fillna('?')
    t1_r = (merged['heldout_performance'].max() - merged['heldout_performance'].min()) or 1
    t2_r = (merged[t2_col].max()                - merged[t2_col].min())               or 1
    merged['_t1_norm'] = (merged['heldout_performance'] - merged['heldout_performance'].min()) / t1_r
    merged['_t2_norm'] = (merged[t2_col]                - merged[t2_col].min())               / t2_r
    merged['combined_score'] = (merged['_t1_norm'] + merged['_t2_norm']) / 2
    return merged


full_df        = _build_full_frame(ai_df_thr,    t2_col=T2_PRIMARY, t2_df=track2_df)
full_df_normal = _build_full_frame(ai_df_normal, t2_col=T2_PRIMARY, t2_df=track2_df)
print(f'\nFull frame (heldout threshold + T2 + arch): n = {len(full_df)}')
print(f'Full frame (Normal  threshold + T2 + arch): n = {len(full_df_normal)}')

---
## Main figure panels

The cells below generate Figures 3B, 3C, and 3D.
Each cell is self-contained: it reads from the already-loaded DataFrames,
prints a short sanity check, and saves PDF + PNG.

### Fig 3B — Mouse vs AI held-out performance histogram

Distribution of all AI submissions' held-out performance (mean over Clutter,
Contrast, RDK). The mouse mean ± SEM is shown as a vertical line with shaded band.

In [ ]:
_ai_held    = ai_df['heldout_performance'].dropna().values
_mouse_held = mouse_df['heldout_performance'].dropna().values
_m_mean     = float(_mouse_held.mean())
_m_sem      = float(stats.sem(_mouse_held)) if len(_mouse_held) > 1 else 0.0

# ── Sanity ────────────────────────────────────────────────────────────────
print('Fig 3B sanity:')
print(f'  AI    : n={len(_ai_held)}, mean={_ai_held.mean():.4f}, '
      f'sem={stats.sem(_ai_held):.4f}, median={np.median(_ai_held):.4f}')
print(f'  Mouse : n={len(_mouse_held)}, mean={_m_mean:.4f}, sem={_m_sem:.4f}')
_frac = float((_ai_held > _m_mean).mean())
print(f'  Fraction AI > mouse mean: {_frac:.3f} '
      f'({int((_ai_held > _m_mean).sum())}/{len(_ai_held)})')

# ── Plot ──────────────────────────────────────────────────────────────────
with plt.rc_context(PAPER_RC):
    fig, ax = plt.subplots(figsize=(9*CM, 6.5*CM))

    ax.hist(_ai_held, bins=25, color='#555555', alpha=0.85,
            edgecolor='black', linewidth=0.4,
            label=f'AI submissions (n={len(_ai_held)})')
    ax.axvspan(_m_mean - _m_sem, _m_mean + _m_sem,
               color=COLOR_MOUSE, alpha=0.25, zorder=1)
    ax.axvline(_m_mean, color=COLOR_MOUSE, linewidth=2.0, zorder=4,
               label=f'Mouse mean ± SEM  (n={len(_mouse_held)})')

    ax.set_xlabel('Held-out performance')
    ax.set_ylabel('Number of models')
    ax.set_title('Held-out robustness relative to mice')
    ax.legend(frameon=False, loc='upper left', fontsize=6)

    # Fixed tick positions: x = performance [0, 0.5, 1]; y = counts [0, 10, 20, 30]
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 10, 20, 30])

    style_ax(ax)
    fig.tight_layout()
    save_fig(fig, 'fig3b_mouse_vs_ai_heldout_hist', 'main')
    plt.show()


### Fig 3C — Architecture robustness

Per-backbone family: individual model points (jittered, gray) plus black mean ± SEM marker.
Backbones with n < 3 models are shown dimmed with an asterisk.

In [ ]:
def _arch_panel(df, metric, ylabel, title, fname, subdir):
    """Per-backbone: jittered points + mean ± SEM error bar.  n<3 dimmed."""
    grouped   = df.dropna(subset=[metric]).groupby('backbone')
    backbones = sorted(grouped.groups,
                       key=lambda b: -grouped.get_group(b)[metric].mean())
    rng = np.random.default_rng(seed=0)

    with plt.rc_context(PAPER_RC):
        fig, ax = plt.subplots(figsize=(13*CM, 6.5*CM))
        means, sems, ns = [], [], []
        for i, bb in enumerate(backbones):
            v  = grouped.get_group(bb)[metric].values
            n  = len(v)
            m_ = float(np.mean(v))
            s_ = float(stats.sem(v)) if n > 1 else 0.0
            means.append(m_); sems.append(s_); ns.append(n)
            dim = n < 3
            jitter = rng.uniform(-0.13, 0.13, size=n)
            ax.scatter(np.full(n, i) + jitter, v,
                       color=COLOR_GRAY if not dim else 'lightgray',
                       alpha=0.55 if not dim else 0.35,
                       edgecolor='none', s=14, zorder=2)
            ax.errorbar([i], [m_], yerr=[s_], fmt='o',
                        color='black', markerfacecolor='black',
                        capsize=3.5, markersize=4.5, lw=1.0, zorder=3)
            if dim:
                ax.text(i, m_ + s_ + 0.025 * max(abs(v.max()), 0.05),
                        '*', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_xticks(range(len(backbones)))
        ax.set_xticklabels(backbones, rotation=30, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        # n annotation below each backbone label
        for i, n in enumerate(ns):
            ax.annotate(f'n={n}', xy=(i, ax.get_ylim()[0]),
                        xytext=(0, -22), textcoords='offset points',
                        ha='center', fontsize=5.5, color=COLOR_GRAY)
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
        style_ax(ax)
        fig.tight_layout()
        save_fig(fig, fname, subdir)
        plt.show()

    # Sanity
    print(f'  {fname} ({metric}) per backbone:')
    print(pd.DataFrame({'backbone': backbones, 'n': ns,
                        'mean': [f'{m:.4f}' for m in means],
                        'sem':  [f'{s:.4f}' for s in sems]})
          .to_string(index=False))
    _groups = [grouped.get_group(bb)[metric].values
               for bb in backbones if len(grouped.get_group(bb)) >= 2]
    if len(_groups) >= 2:
        _f, _p = stats.f_oneway(*_groups)
        print(f'  One-way ANOVA across backbones: F={_f:.3f}, p={_p:.3g}')


print('Fig 3C:')
_arch_panel(ai_arch, 'heldout_performance',
           ylabel='Held-out performance',
           title='Architecture: held-out robustness  (* n<3)',
           fname='fig3c_architecture_heldout', subdir='main')


---
## Supplementary panels

The cells below generate all supplementary figures.

### Supp — All Track 1 perturbations vs Track 2

1 × 5 grid showing the scatter of each perturbation condition against neural alignment.

In [ ]:
# Merge T1 + T2 (all models, no threshold) for the all-perturbations scatter
_merged_t1t2 = ai_df.merge(track2_df[['id', T2_PRIMARY]], on='id', how='inner')

_rows_all = []
with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(1, 5, figsize=(20*CM, 5*CM), sharey=True)
    for ax, _pert in zip(axes, PERTURBATIONS):
        _sub = _merged_t1t2[[_pert, T2_PRIMARY]].dropna()
        if len(_sub) >= 3:
            _x, _y = _sub[_pert].values, _sub[T2_PRIMARY].values
            ax.scatter(_x, _y, color=COLOR_AI, alpha=0.6,
                       edgecolor='black', linewidth=0.3, s=12)
            _sl, _ic, _r, _pv, _ = stats.linregress(_x, _y)
            _xfit = np.linspace(_x.min(), _x.max(), 100)
            ax.plot(_xfit, _sl*_xfit + _ic, '--', color='black', lw=1.0)
            corr_annotation(ax, _r, _pv, len(_sub), loc='upper left')
            _rows_all.append([_pert, _r, _pv, len(_sub)])
        else:
            ax.text(0.5, 0.5, f'n={len(_sub)}', transform=ax.transAxes,
                    ha='center', va='center')
            _rows_all.append([_pert, np.nan, np.nan, len(_sub)])
        ax.set_xlabel(_pert)
        ax.set_title(_pert, fontsize=7)
        max4_ticks(ax)
        style_ax(ax)
    axes[0].set_ylabel(f'Track 2: {T2_PRIMARY}')
    fig.suptitle('Track 1 perturbations vs Track 2 neural alignment',
                 fontsize=8, y=1.02)
    fig.tight_layout()
    save_fig(fig, 'supp_track1_track2_all_scatters', 'supplement')
    plt.show()

print('Supp all-scatters sanity:')
print(pd.DataFrame(_rows_all, columns=['perturbation', 'r', 'p', 'n'])
      .to_string(index=False))

### Supp — Recurrence effect

Compares held-out performance and performance drop between models with and without
recurrent connections. Points are jittered individual models; black = mean ± SEM.

In [ ]:
def _recurrence_panel(ax, df, metric, ylabel, title):
    sub   = df.dropna(subset=[metric])
    no_v  = sub[~sub['has_recurrence_bool']][metric].values
    rec_v = sub[ sub['has_recurrence_bool']][metric].values
    rng   = np.random.default_rng(seed=0)
    for i, (vals, color) in enumerate(
            [(no_v, COLOR_GRAY), (rec_v, COLOR_RECURRENCE)]):
        _j = rng.uniform(-0.13, 0.13, size=len(vals))
        ax.scatter(np.full(len(vals), i) + _j, vals,
                   color=color, alpha=0.55, edgecolor='none', s=16, zorder=2)
    _means = [float(np.mean(no_v)), float(np.mean(rec_v))]
    _sems  = [float(stats.sem(no_v)) if len(no_v)  > 1 else 0.0,
              float(stats.sem(rec_v)) if len(rec_v) > 1 else 0.0]
    _ns    = [len(no_v), len(rec_v)]
    for i, (m_, s_) in enumerate(zip(_means, _sems)):
        ax.errorbar([i], [m_], yerr=[s_], fmt='o',
                    color='black', markerfacecolor='black',
                    capsize=3.5, markersize=5, lw=1.1, zorder=3)
    if len(no_v) > 1 and len(rec_v) > 1:
        _t, _pt = stats.ttest_ind(no_v, rec_v, equal_var=False)
        ax.text(0.5, 0.98, f't = {_t:.2f},  p = {_pt:.3g}',
                transform=ax.transAxes, ha='center', va='top', fontsize=6,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                          alpha=0.9, edgecolor='lightgray', linewidth=0.5))
    else:
        _t = _pt = np.nan
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'No recurrence\n(n={_ns[0]})',
                        f'Recurrence\n(n={_ns[1]})'])
    ax.set_xlim(-0.55, 1.55)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    style_ax(ax)
    return {'metric': metric,
            'no_rec_n': _ns[0], 'no_rec_mean': _means[0], 'no_rec_sem': _sems[0],
            'rec_n': _ns[1], 'rec_mean': _means[1], 'rec_sem': _sems[1],
            't': _t, 'p_t': _pt}


with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(1, 2, figsize=(14*CM, 6.5*CM))
    _rh = _recurrence_panel(axes[0], ai_rec, 'heldout_performance',
                            'Held-out performance', 'Held-out: recurrence')
    _rd = _recurrence_panel(axes[1], ai_rec, 'performance_drop',
                            'Performance drop', 'Drop: recurrence')
    fig.tight_layout()
    save_fig(fig, 'supp_recurrence_effect', 'supplement')
    plt.show()

print('Supp recurrence sanity:')
for _rr in [_rh, _rd]:
    print(f'  [{_rr["metric"]}]  '
          f'no-rec: n={_rr["no_rec_n"]}, mean={_rr["no_rec_mean"]:.4f}  |  '
          f'rec: n={_rr["rec_n"]}, mean={_rr["rec_mean"]:.4f}  |  '
          f'Welch t={_rr["t"]:.3f}, p={_rr["p_t"]:.3g}')


### Fig 3D / Supp — Track 1 × Track 2 scatter (four variants)

Four versions are produced by the cell below.

**Main panels** — T1 winner and T2 winner highlighted; no all-rounders:
- `fig3d_track1_track2_scatter_winners_heldout_thresh` — held-out ≥ 0.5 filter
- `fig3d_track1_track2_scatter_winners_normal_thresh` — Normal ≥ 0.5 filter

**Supplement panels** — same plus all-rounders highlighted:
- `supp_track1_track2_scatter_winners_allrounders_heldout_thresh`
- `supp_track1_track2_scatter_winners_allrounders_normal_thresh`

In [ ]:
def _scatter_winners(df_full, t2_col=T2_PRIMARY,
                     show_allrounders=False,
                     fname='scatter', subdir='supplement',
                     filter_label=None):
    """Track 1 held-out vs Track 2 scatter with highlighted top models.

    Parameters
    ----------
    show_allrounders : bool
        If True, also highlight the top model by combined (T1+T2) normalised score.
    fname : str
        Output filename stem (no extension).
    subdir : 'main' or 'supplement'
        Output sub-directory under figures/.
    filter_label : str or None
        Short description of the T1 filter shown in the title.
    """
    _x = df_full['heldout_performance'].values
    _y = df_full[t2_col].values
    _r, _p = stats.pearsonr(_x, _y)
    _sl, _ic, _, _, _ = stats.linregress(_x, _y)

    _grp_specs = [
        ('heldout_performance', 'Best T1', '#E07B39', 'o'),
        (t2_col,               'Best T2',  COLOR_AI, 's'),
    ]
    if show_allrounders:
        _grp_specs.append(('combined_score', 'Best overall', '#2CA02C', 'D'))

    _title = 'Track 1 vs Track 2'
    if filter_label:
        _title += f'\n({filter_label})'

    with plt.rc_context(PAPER_RC):
        fig, ax = plt.subplots(figsize=(7*CM, 7*CM))   # square
        fig.subplots_adjust(bottom=0.20)

        # Background cloud — light grey
        ax.scatter(_x, _y, color='#888888', alpha=0.75,
                   edgecolor='none', s=16, zorder=2)

        # Regression line
        _xfit = np.linspace(_x.min(), _x.max(), 100)
        ax.plot(_xfit, _sl*_xfit + _ic, '--',
                color='black', lw=0.9, alpha=0.40, zorder=2)

        # Highlighted groups — one point per group
        _annotated_ids = {}   # id → label, to skip duplicate annotations
        for _sc, _lbl, _col_h, _mk in _grp_specs:
            _row = df_full.nlargest(1, _sc).iloc[0]
            _hx, _hy = _row['heldout_performance'], _row[t2_col]
            ax.scatter([_hx], [_hy],
                       color=_col_h, edgecolor='black', linewidth=0.7,
                       marker=_mk, s=34, zorder=4)
            if _row['id'] not in _annotated_ids:
                ax.annotate(_row['owner'], (_hx, _hy),
                            xytext=(5, 4), textcoords='offset points',
                            fontsize=5.5, color=lighten_color(_col_h, 0.1),
                            ha='left', va='bottom', zorder=5)
                _annotated_ids[_row['id']] = _lbl

        corr_annotation(ax, _r, _p, len(df_full), loc='lower right')

        _leg = ax.legend(
            handles=[mlines.Line2D([], [], marker=_mk, color='w',
                                   markerfacecolor=_col_h, markeredgecolor='black',
                                   markersize=5, label=_lbl)
                     for _, _lbl, _col_h, _mk in _grp_specs],
            loc='upper center', bbox_to_anchor=(0.5, -0.13),
            ncol=len(_grp_specs), frameon=True, facecolor='white',
            edgecolor='#cccccc', fontsize=6, handlelength=1.0,
            borderpad=0.4, columnspacing=0.8, handletextpad=0.4)
        _leg.get_frame().set_linewidth(0.5)

        max4_ticks(ax)
        ax.set_xlabel('Track 1: held-out performance')
        ax.set_ylabel(f'Track 2: {t2_col}')
        ax.set_title(_title)
        style_ax(ax)
        save_fig(fig, fname, subdir)
        plt.show()

    # Sanity prints
    print(f'\n{subdir}/{fname}:  n = {len(df_full)},  r = {_r:.4f},  p = {_p:.3g}')
    for _sc, _lbl, _, _ in _grp_specs:
        _row = df_full.nlargest(1, _sc).iloc[0]
        print(f'  {_lbl:<16}: {_row["owner"]:<22}  '
              f'heldout = {_row["heldout_performance"]:.3f},  '
              f'{t2_col} = {_row[t2_col]:.4f}')


# ── A: Main — held-out threshold, T1/T2 winners only ─────────────────────
_scatter_winners(full_df, t2_col=T2_PRIMARY,
                 show_allrounders=False,
                 fname='fig3d_track1_track2_scatter_winners_heldout_thresh',
                 subdir='main',
                 filter_label=f'held-out ≥ {T1_THRESHOLD}')

# ── B: Main — Normal threshold, T1/T2 winners only ───────────────────────
_scatter_winners(full_df_normal, t2_col=T2_PRIMARY,
                 show_allrounders=False,
                 fname='fig3d_track1_track2_scatter_winners_normal_thresh',
                 subdir='main',
                 filter_label=f'Normal ≥ {T1_NORMAL_THRESHOLD}')

# ── C: Supplement — held-out threshold, winners + all-rounders ───────────
_scatter_winners(full_df, t2_col=T2_PRIMARY,
                 show_allrounders=True,
                 fname='supp_track1_track2_scatter_winners_allrounders_heldout_thresh',
                 subdir='supplement',
                 filter_label=f'held-out ≥ {T1_THRESHOLD}')

# ── D: Supplement — Normal threshold, winners + all-rounders ─────────────
_scatter_winners(full_df_normal, t2_col=T2_PRIMARY,
                 show_allrounders=True,
                 fname='supp_track1_track2_scatter_winners_allrounders_normal_thresh',
                 subdir='supplement',
                 filter_label=f'Normal ≥ {T1_NORMAL_THRESHOLD}')


### Supp — Track 1 × Track 2 colored by architecture

Left panel: colored by number of parameters (log scale, viridis).
Right panel: colored by model depth (linear scale, plasma).

### Supp — Model-property trends

2 × 3 composite figure. Each column is a model property (training steps, # parameters, # layers);
each row is a performance metric (Track 1 held-out, Track 2 neural alignment).
Pearson r is computed between log₁₀(x) and y for parameters, and between x and y for steps/depth.
A dashed regression line is added to panels where the correlation is significant (p < 0.05).
Training steps are extracted from ONNX filenames in `model_metadata.csv` (140/236 models have parseable values).

In [ ]:
# ── Merge T1 / T2 with architecture metadata (incl. training_steps) ───────
_arch_mp_cols = ['id', 'num_params', 'num_layers']
if 'training_steps' in arch_df.columns:
    _arch_mp_cols.append('training_steps')
_arch_mp = arch_df[_arch_mp_cols].copy()
_mp_t1   = ai_df.merge(_arch_mp,     on='id', how='inner')
_mp_t2   = track2_df.merge(_arch_mp, on='id', how='inner')

# ── Column mapping ─────────────────────────────────────────────────────────
# Training steps: parsed from ONNX filenames stored in model_metadata.csv.
_STEPS_COL = None
for _cand in ('training_steps', 'num_steps', 'steps', 'train_steps'):
    if _cand in arch_df.columns:
        _STEPS_COL = _cand
        break
if _STEPS_COL is None:
    warnings.warn(
        'No training-steps column found in model_metadata.csv — '
        'panels A/D will show "not available".',
        stacklevel=2)

_PARAM_COL = 'num_params'
_DEPTH_COL = 'num_layers'

print('Supp model-property trends — column mapping:')
print(f'  training_steps : '
      f'{"NOT AVAILABLE — panels A/D blank" if _STEPS_COL is None else _STEPS_COL}')
print(f'  parameters     : {_PARAM_COL}')
print(f'  depth          : {_DEPTH_COL}')
print(f'\n  T1 + metadata  : {len(_mp_t1)} models')
print(f'  T2 + metadata  : {len(_mp_t2)} models')
if _STEPS_COL:
    _n_steps_t1 = _mp_t1[_STEPS_COL].notna().sum()
    _n_steps_t2 = _mp_t2[_STEPS_COL].notna().sum()
    print(f'  T1 with training steps : {_n_steps_t1}')
    print(f'  T2 with training steps : {_n_steps_t2}')


# ── Per-panel Pearson r + regression (log10(x) if log_x) ──────────────────
def _prop_stats(df, xcol, ycol, log_x=False):
    """Returns (sub, r, p, slope, intercept).
    x-values are log10-transformed before regression when log_x=True,
    so slope/intercept apply in log-space (for drawing the line)."""
    sub = df[[xcol, ycol]].dropna()
    if log_x:
        sub = sub[sub[xcol] > 0]
    if len(sub) < 3:
        return sub, np.nan, np.nan, np.nan, np.nan
    x = np.log10(sub[xcol].values) if log_x else sub[xcol].values
    y = sub[ycol].values
    r, p = stats.pearsonr(x, y)
    slope, intercept, _, _, _ = stats.linregress(x, y)
    return sub, r, p, slope, intercept


# Panel spec: (xcol, ycol, log_x, df, label, x-display-name)
_prop_panels = [
    (_STEPS_COL,  'heldout_performance', False, _mp_t1, 'A', 'Training steps'),
    (_PARAM_COL,  'heldout_performance', True,  _mp_t1, 'B', '# parameters'),
    (_DEPTH_COL,  'heldout_performance', False, _mp_t1, 'C', '# layers'),
    (_STEPS_COL,  T2_PRIMARY,            False, _mp_t2, 'D', 'Training steps'),
    (_PARAM_COL,  T2_PRIMARY,            True,  _mp_t2, 'E', '# parameters'),
    (_DEPTH_COL,  T2_PRIMARY,            False, _mp_t2, 'F', '# layers'),
]

print('\n  Per-panel statistics:')
for xcol, ycol, log_x, df, lbl, xname in _prop_panels:
    if xcol is None:
        print(f'    Panel {lbl} ({xname} vs {ycol}): NOT AVAILABLE')
        continue
    sub, r, p, *_ = _prop_stats(df, xcol, ycol, log_x)
    xdesc = f'log10({xcol})' if log_x else xcol
    sig = ' *' if (not np.isnan(p) and p < 0.05) else ''
    print(f'    Panel {lbl}: n={len(sub)},  r={r:+.4f},  p={p:.3g}{sig}'
          f'  ({xdesc} vs {ycol})')

# ── Figure ────────────────────────────────────────────────────────────────
_YLABELS = {
    'heldout_performance': 'Held-out performance',
    T2_PRIMARY:            f'Track 2: {T2_PRIMARY}',
}

with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(2, 3, figsize=(17.8*CM, 9.0*CM))
    fig.subplots_adjust(hspace=0.52, wspace=0.38)

    for ax, (xcol, ycol, log_x, df, lbl, xname) in zip(axes.flat, _prop_panels):
        # Panel label (bold, top-left)
        ax.text(0.03, 0.97, lbl, transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='top', ha='left')
        ax.set_xlabel(xname)
        ax.set_ylabel(_YLABELS.get(ycol, ycol))
        style_ax(ax)

        if xcol is None:
            # Training steps not available — blank panel with note
            ax.text(0.5, 0.45, 'Training steps\nnot available',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=6, color='#999999', style='italic')
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        sub, r, p, slope, intercept = _prop_stats(df, xcol, ycol, log_x)
        if len(sub) < 3:
            ax.text(0.5, 0.5, f'n = {len(sub)}\n(insufficient)',
                    transform=ax.transAxes, ha='center', va='center', fontsize=6)
            continue

        ax.scatter(sub[xcol].values, sub[ycol].values,
                   color=COLOR_AI, alpha=0.50, edgecolor='none', s=10, zorder=2)

        if log_x:
            ax.set_xscale('log')
        else:
            ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

        # Regression line — only when p < 0.05
        if not np.isnan(p) and p < 0.05:
            if log_x:
                _xlog  = np.log10(sub[xcol].values)
                _xline = np.linspace(_xlog.min(), _xlog.max(), 100)
                ax.plot(10 ** _xline, slope * _xline + intercept,
                        color='#444444', lw=0.9, ls='--', alpha=0.7, zorder=3)
            else:
                _xlin  = sub[xcol].values
                _xline = np.linspace(_xlin.min(), _xlin.max(), 100)
                ax.plot(_xline, slope * _xline + intercept,
                        color='#444444', lw=0.9, ls='--', alpha=0.7, zorder=3)

        corr_annotation(ax, r, p, len(sub),
                        loc='lower right' if r >= 0 else 'upper right')

    # Column titles (top row only)
    axes[0, 0].set_title('Training steps', fontsize=7)
    axes[0, 1].set_title('# parameters',   fontsize=7)
    axes[0, 2].set_title('# layers',       fontsize=7)

    save_fig(fig, 'supp_model_property_trends', 'supplement')
    plt.show()

---
## Summary

In [ ]:
print('=' * 66)
print('  04_figures.ipynb — complete')
print('=' * 66)

# ── Dataset counts ────────────────────────────────────────────────────────
n_t1     = len(ai_df)
n_t2     = len(track2_df)
n_both   = len(set(ai_df['id']) & set(track2_df['id']))
n_arch   = len(ai_arch)
n_mouse  = len(mouse_df)
n_neuron = int(track2_df['n_neurons'].iloc[0])

print(f'\n  Track 1 models              : {n_t1}')
print(f'  Track 2 models              : {n_t2}')
print(f'  Models with T1 and T2       : {n_both}')
print(f'  Models with arch metadata   : {n_arch}')
print(f'  Mouse sessions              : {n_mouse}')
print(f'\n  Primary T2 column           : {T2_PRIMARY}')
print(f'  Neuron filter (C0.005)      : {n_neuron} neurons')
print(f'\n  Scatter panel model counts:')
print(f'    held-out ≥ {T1_THRESHOLD}  →  n = {len(full_df)}  (heldout-thr variant)')
print(f'    Normal   ≥ {T1_NORMAL_THRESHOLD}  →  n = {len(full_df_normal)}  (normal-thr variant)')

print(f'\n  Backbone distribution:')
print(bb_counts.to_string())

# ── File existence check ───────────────────────────────────────────────────
_expected = [
    # Main panels
    FIG_DIR / 'main' / 'fig3b_mouse_vs_ai_heldout_hist.pdf',
    FIG_DIR / 'main' / 'fig3c_architecture_heldout.pdf',
    FIG_DIR / 'main' / 'fig3d_track1_track2_scatter_winners_heldout_thresh.pdf',
    FIG_DIR / 'main' / 'fig3d_track1_track2_scatter_winners_normal_thresh.pdf',
    # Supplementary
    FIG_DIR / 'supplement' / 'supp_track1_track2_all_scatters.pdf',
    FIG_DIR / 'supplement' / 'supp_recurrence_effect.pdf',
    FIG_DIR / 'supplement' / 'supp_track1_track2_scatter_winners_allrounders_heldout_thresh.pdf',
    FIG_DIR / 'supplement' / 'supp_track1_track2_scatter_winners_allrounders_normal_thresh.pdf',
    FIG_DIR / 'supplement' / 'supp_track1_track2_scatter_arch.pdf',
    FIG_DIR / 'supplement' / 'supp_track1_track2_scatter_arch_normal_thr.pdf',
    FIG_DIR / 'supplement' / 'supp_model_property_trends.pdf',
]
_missing = [p for p in _expected if not p.exists()]

print(f'\n  Saved {len(saved_files)} files  ({len(saved_files)//2} panels):')
_main = sorted(set(Path(f).stem for f in saved_files if 'main' in f))
_supp = sorted(set(Path(f).stem for f in saved_files if 'supplement' in f))
print(f'\n  Main ({len(_main)} panels):')
for _fp in _main:
    print(f'    {_fp}')
print(f'\n  Supplement ({len(_supp)} panels):')
for _fp in _supp:
    print(f'    {_fp}')

# ── Required data-files check ─────────────────────────────────────────────
_data_files = {
    'leaderboard_track1.csv': (DATA_DIR / 'leaderboard_track1.csv', 225),
    'leaderboard_track2.csv': (DATA_DIR / 'leaderboard_track2.csv', 139),
    'model_metadata.csv':     (DATA_DIR / 'model_metadata.csv',     236),
    'mouse_performance.csv':  (DATA_DIR / 'mouse_performance.csv',    5),
}
print(f'\n  Data files:')
for _name, (_path, _expected_rows) in _data_files.items():
    _exists = _path.exists()
    _nrows  = len(pd.read_csv(_path)) if _exists else -1
    _status = '✓' if _exists else '✗'
    print(f'    {_status} {_name:<35} {_nrows} rows  (expected {_expected_rows})')

if _missing:
    print(f'\n⚠  {len(_missing)} expected PDF(s) not found:')
    for _p in _missing:
        print(f'    {_p}')
    raise RuntimeError('Not all expected figure files were produced — see list above.')
else:
    print(f'\n✓  All {len(_expected)} expected PDFs present.')

---
## Winning teams


In [ ]:
# ── Winning teams summary ────────────────────────────────────────────────────────
# Merge Track 1 (ai_df) with Track 2 scores; build id_name string
_wt_t1 = ai_df[["id", "owner", "track1_score", "Normal"]].copy()
_wt_t2 = track2_df[["id", T2_PRIMARY]].copy()
_wt    = _wt_t1.merge(_wt_t2, on="id", how="left")
_wt["id_name"] = _wt["id"].astype(str) + "_" + _wt["owner"].fillna("?")

# ── Top 3 Track 1 winners (by track1_score) ───────────────────────────
_top3_t1 = (
    _wt.dropna(subset=["track1_score"])
       .sort_values("track1_score", ascending=False)
       .drop_duplicates(subset=["owner"], keep="first")
       .head(3)
)

print("Top 3 Track 1 winners (by track1_score):")
print(f"  {'id_name':<40} {'track1_score':>12} {'track2_score':>12}")
print(f"  {'-'*40} {'-'*12} {'-'*12}")
for _, _row in _top3_t1.iterrows():
    _t2 = f"{_row[T2_PRIMARY]:.4f}" if pd.notna(_row[T2_PRIMARY]) else "N/A"
    print(f"  {_row['id_name']:<40} {_row['track1_score']:>12.4f} {_t2:>12}")

print()

# ── Top 3 Track 2 winners (Normal > 0.5, by track2_score) ─────────────────
_top3_t2 = (
    _wt[_wt["Normal"] > 0.5]
       .dropna(subset=[T2_PRIMARY])
       .sort_values(T2_PRIMARY, ascending=False)
       .drop_duplicates(subset=["owner"], keep="first")
       .head(3)
)

print(f"Top 3 Track 2 winners (Normal > 0.5, by {T2_PRIMARY}):")
print(f"  {'id_name':<40} {'track1_score':>12} {'track2_score':>12}")
print(f"  {'-'*40} {'-'*12} {'-'*12}")
for _, _row in _top3_t2.iterrows():
    _t1 = f"{_row['track1_score']:.4f}" if pd.notna(_row['track1_score']) else "N/A"
    print(f"  {_row['id_name']:<40} {_t1:>12} {_row[T2_PRIMARY]:>12.4f}")
